---

## **Part 1: LangChain Framework – Core Architecture (Agentic Use Case Perspective)**

### **The Problem LangChain Solves**
LLMs (like GPT-4) are great at generating text, but:
- They can’t access tools like search, calculator, or your company data.
- They forget everything between messages unless you give them memory.
- They can't reason step-by-step or make decisions dynamically.

**LangChain solves this by acting as a bridge.**

---

### **LangChain Key Components (Agentic Perspective)**

| Component | What It Does | Real Example |
|----------|--------------|--------------|
| **LLM** | Connects to GPT-4, Claude, Mistral, etc. | "Write an email to HR" |
| **PromptTemplate** | Creates reusable prompt structures | "Write a tweet about {topic}" |
| **Tool** | External actions (Search, API call, DB) | Weather lookup, PDF reader |
| **Agent** | Decides *when* and *how* to use tools | "Hmm… to answer this, I need weather + location" |
| **Memory** | Stores past conversations | Multi-turn chatbots |
| **Retriever** | Retrieves documents from a knowledge base | Vector search in RAG |
| **Chain** | Links LLM calls together in steps | Extract from PDF → Generate summary → Send email |

---

### **Visual Architecture: Simple Agentic System**

```
[User Query]
     |
     v
[LangChain Agent] --- decides --->
     |                           |
     |                          [Tool 1: Weather API]
     |                          [Tool 2: Flight Search]
     |                          [Tool 3: Calculator]
     |
[Final Answer to User]
```


---


---

### **Problem Statement**

In many real-world scenarios, users seek quick, accurate answers to mathematical expressions embedded within natural language queries. While large language models like GPT-4 can reason over numbers, they are not inherently designed to perform precise arithmetic or complex calculations reliably. This often leads to incorrect or imprecise answers when users ask math-related questions.

To overcome this, we aim to build an intelligent assistant that can:
- Understand the user’s natural language input.
- Identify embedded mathematical expressions.
- Automatically invoke a reliable calculator tool to compute the result.
- Return the final answer in a natural, conversational format.

This problem becomes particularly important in domains such as finance, e-commerce, education, or productivity tools—where accuracy in mathematical interpretation is critical.

---

### **Objective**

Create a LangChain-powered agent that integrates a calculator tool using OpenAI’s `gpt-4o` model. The assistant should:
- Detect when a math expression needs tool support.
- Call the calculator tool automatically.
- Handle the reasoning flow using LangChain's tool-calling framework.
- Maintain proper execution history using `agent_scratchpad`.

---

In [184]:
# Imports
from langchain.agents import create_tool_calling_agent, AgentExecutor

from langchain.tools import tool

from langchain_openai import ChatOpenAI
# Gemini Models: from langchain_google_palm import ChatGooglePalm
# Ollama Models: from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from dotenv import load_dotenv
import os

In [185]:
load_dotenv()

True

In [186]:
# eval() - This function is used to evaluate a string expression and return the result.

print(eval("(4.5732 *52) + (10.120 / 4) - 10"))
print(eval("(43132 *52) + (10.1220 / 124) - 10 + 43"))

230.3364
2242897.0816290323


In [187]:
# Print my battery percentage
import psutil
battery = psutil.sensors_battery()
print(f"Battery percentage: {battery.percent}%")
# Print if my laptop is plugged in or not
print(f"Laptop plugged in: {battery.power_plugged}")

Battery percentage: 68%
Laptop plugged in: True


In [188]:
# Print system name and OS
import platform
system_name = platform.system()
os_name = platform.platform()
print(f"System Name: {system_name}")
print(f"OS Name: {os_name}")

System Name: Darwin
OS Name: macOS-15.5-arm64-arm-64bit


In [189]:
# Print current date and time
from datetime import datetime
current_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Current Date and Time: {current_date}")

Current Date and Time: 2025-06-28 14:54:00


In [190]:
!pip install openai langchain langchain-community python-dotenv -q
# langchain-community - A community-driven extension of LangChain that includes additional tools and features. It has all the third-party tools.

In [191]:
!pip install google-search-results -q

In [192]:
# External API utilities from LangChain Community
from langchain_community.utilities.serpapi import SerpAPIWrapper

In [193]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")

In [194]:
# Step 3: Set up SerpAPI (Google Search wrapper)
search_tool = SerpAPIWrapper(
    serpapi_api_key=SERPAPI_API_KEY
)

In [195]:
from langchain.tools import Tool

In [196]:
WEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

In [212]:
import requests
import json

In [221]:
@tool
def calculator(expression: str) -> float:
    """A simple calculator function that evaluates a mathematical expression."""
    return eval(expression)

@tool
def get_battery_percentage() -> str:
    """Get the current battery percentage of the laptop."""
    battery = psutil.sensors_battery()
    if battery is None:
        return "Battery information is not available."
    if battery.percent > 80:
        return f"Battery percentage is {battery.percent}%. You have enough battery. Right now, your laptop is {'plugged in' if battery.power_plugged else 'not plugged in'}."
    elif battery.percent > 50:
        return f"Battery percentage is {battery.percent}%. You have a decent amount of battery left. Right now, your laptop is {'plugged in' if battery.power_plugged else 'not plugged in'}."
    elif battery.percent > 20:
        return f"Battery percentage is {battery.percent}%. You should consider charging soon. Right now, your laptop is {'plugged in' if battery.power_plugged else 'not plugged in'}."
    else:
        return f"Battery percentage is {battery.percent}%. Your battery is low! Please charge your laptop soon. Right now, your laptop is {'plugged in' if battery.power_plugged else 'not plugged in'}."
    
@tool
def get_system_name() -> str:
    """Get the name of the system."""
    return platform.system()

@tool
def get_os_name() -> str:
    """Get the name of the operating system."""
    return platform.platform()

@tool
def get_current_date_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
# ----------------------------
# Define actual weather tool
# ----------------------------
def get_current_weather(location: str, unit: str = "metric") -> str:
    """
    Calls OpenWeatherMap API to fetch current weather.
    """
    try:
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q": location,
            "appid": WEATHER_API_KEY,
            "units": unit
        }
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        print("=="*30)  # Debugging line to indicate successful API call
        print(data)
        print("=="*30)  # Debugging line to indicate successful API call

        result = {
            "location": f"{data['name']}, {data['sys']['country']}",
            "temperature": data['main']['temp'],
            "condition": data['weather'][0]['description'],
            "unit": "°C" if unit == "metric" else "°F"
        }
        return json.dumps(result)

    except Exception as e:
        return f"Error: {str(e)}"

tools = [calculator, get_battery_percentage, get_system_name, get_os_name, get_current_date_time,
         Tool.from_function(
                func=search_tool.run,
                name="google_search",
                description="Use this tool to answer questions about current events, find information on the web, or search for specific topics. It can be used to retrieve information from the internet, including news articles, blog posts, and other online content."
         ),
         get_current_weather
         ]

In [222]:
llm = ChatOpenAI(model="gpt-4o")

In [223]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. You can answer questions about the current date and time, system information, battery status, and perform calculations. You can also search the web for information. You will always start by justifying how you came to a particular answer step-by-step to the user, and then finally print the actual answer. Your responses should have foillowing format: 'I found that the answer is _____. This is because _____.'"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [224]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [225]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [202]:
# # Invoke the agent executor with a sample input
# response = agent_executor.invoke({"input": "What is the capital of France?"})

# # Print the response
# print(response)

In [203]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "What is the capital of France?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...
I found that the answer is Paris. This is because Paris is widely recognized as the capital city of France, serving as its political, cultural, and economic center.

> Finished chain.
I found that the answer is Paris. This is because Paris is widely recognized as the capital city of France, serving as its political, cultural, and economic center.


In [176]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "What is the  boiling point of water?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...
The boiling point of water is 100 degrees Celsius or 212 degrees Fahrenheit at standard atmospheric pressure (1 atmosphere or 101.3 kPa). I found that the answer is 100 degrees Celsius or 212 degrees Fahrenheit because this is a well-documented physical property of water under normal atmospheric conditions.

> Finished chain.
The boiling point of water is 100 degrees Celsius or 212 degrees Fahrenheit at standard atmospheric pressure (1 atmosphere or 101.3 kPa). I found that the answer is 100 degrees Celsius or 212 degrees Fahrenheit because this is a well-documented physical property of water under normal atmospheric conditions.


In [11]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "explain the process of photosynthesis in two lines as if you're explaining it to a 10 year old child"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...
Photosynthesis is when plants use sunlight to turn water and air into food and oxygen. It's like the plant's way of making its own snacks using sunlight!

> Finished chain.
Photosynthesis is when plants use sunlight to turn water and air into food and oxygen. It's like the plant's way of making its own snacks using sunlight!


In [37]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "What is (4.5732 *52) + (10.120 / 4) - 10?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '4.5732 * 52'}`


237.8064
Invoking: `calculator` with `{'expression': '10.120 / 4'}`


2.53
Invoking: `calculator` with `{'expression': '237.8064 + 2.53 - 10'}`


230.3364The result of \((4.5732 \times 52) + \left(\frac{10.120}{4}\right) - 10\) is 230.3364.

> Finished chain.
The result of \((4.5732 \times 52) + \left(\frac{10.120}{4}\right) - 10\) is 230.3364.


In [13]:
print((45 *2) + (100 / 4) - 10)

105.0


In [15]:
print((4.5732 *52) + (10.120 / 4) - 10)

230.3364


In [48]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "What is my laptop battery percentage"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_battery_percentage` with `{}`


Battery percentage is 37%. You should consider charging soon. Right now, your laptop is plugged in.Your laptop battery is at 37%. It's plugged in, so it's currently charging.

> Finished chain.
Your laptop battery is at 37%. It's plugged in, so it's currently charging.


In [49]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "is my laptop getting charged currently?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_battery_percentage` with `{}`


Battery percentage is 38%. You should consider charging soon. Right now, your laptop is plugged in.Yes, your laptop is currently plugged in and charging. The battery percentage is at 38%.

> Finished chain.
Yes, your laptop is currently plugged in and charging. The battery percentage is at 38%.


In [63]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is the current time?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_date_time` with `{}`


2025-06-28 14:31:18The current time is 14:31 on June 28, 2025.

> Finished chain.
The current time is 14:31 on June 28, 2025.


In [177]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is the current time in 12 hr format?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_date_time` with `{}`


2025-06-28 14:51:17I found that the current time is 2:51 PM. This is because the current 24-hour format time is 14:51, which converts to 2:51 PM in 12-hour format.

> Finished chain.
I found that the current time is 2:51 PM. This is because the current 24-hour format time is 14:51, which converts to 2:51 PM in 12-hour format.


In [133]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is the current time in 12 hr format and what day is it?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_date_time` with `{}`


2025-06-28 14:47:54The current time is 2:47 PM, and today is Saturday, June 28, 2025.

> Finished chain.
The current time is 2:47 PM, and today is Saturday, June 28, 2025.


In [66]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is the system name and OS name?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_system_name` with `{}`


Darwin
Invoking: `get_os_name` with `{}`


macOS-15.5-arm64-arm-64bitThe system name is **Darwin** and the operating system name is **macOS-15.5-arm64-arm-64bit**.

> Finished chain.
The system name is **Darwin** and the operating system name is **macOS-15.5-arm64-arm-64bit**.


In [18]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is today's date?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...
I'm unable to provide real-time information such as the current date. However, you can easily find out today's date by checking your device or using a calendar application.

> Finished chain.
I'm unable to provide real-time information such as the current date. However, you can easily find out today's date by checking your device or using a calendar application.


In [19]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is my System's name"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...
I'm unable to determine the name of your system. However, you can usually find your system's name by checking your operating system's settings or the details in your device's system information. If you're using Windows, you can often find it under "System" in the Control Panel or Settings app. If you're using macOS, you can check under "About This Mac" from the Apple menu. On Linux, you can usually find it in the system settings or by using terminal commands like `hostname`.

> Finished chain.
I'm unable to determine the name of your system. However, you can usually find your system's name by checking your operating system's settings or the details in your device's system information. If you're using Windows, you can often find it under "System" in the Control Panel or Settings app. If you're using macOS, you can check under "About This Mac" from the Apple menu. On Linux, you can usually find it in the system settings or by using terminal command

In [226]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is the current temperature in Singapore?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_weather` with `{'location': 'Singapore'}`


{'coord': {'lon': 103.8501, 'lat': 1.2897}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 30.39, 'feels_like': 37.39, 'temp_min': 30.39, 'temp_max': 30.39, 'pressure': 1008, 'humidity': 79, 'sea_level': 1008, 'grnd_level': 1007}, 'visibility': 10000, 'wind': {'speed': 6.04, 'deg': 189, 'gust': 6.35}, 'clouds': {'all': 98}, 'dt': 1751102308, 'sys': {'country': 'SG', 'sunrise': 1751065312, 'sunset': 1751109212}, 'timezone': 28800, 'id': 1880252, 'name': 'Singapore', 'cod': 200}
{"location": "Singapore, SG", "temperature": 30.39, "condition": "overcast clouds", "unit": "\u00b0C"}I found that the current temperature in Singapore is 30.39°C with overcast clouds. This is because I used the current weather tool to fetch the weather data for Singapore, which reported the temperature and weather conditions

In [108]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is the the gasoline price in Singapore today?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `google_search` with `current gasoline price in Singapore`


['Petrol Prices with Discount ; 95. $2.26. $2.27 ; 98. $2.65. $2.68 ; Premium. -. $2.68 ; Diesel. $2.07. $2.08.', 'Gasoline Prices in Singapore remained unchanged at 2.83 USD/Liter in May. Gasoline Prices in Singapore averaged 1.80 USD/Liter from 1990 until 2025, ...', 'Singapore Retail Prices: Petrol, 95 Octane (Per Litre) data was reported at 2.700 SGD/l in Dec 2024. This stayed constant from the previous number of 2.700 ...', 'Find out the latest Caltex petrol & diesel price here, updated regularly to keep you informed.', 'Singapore, 2.83, 2.83, May/25, USD/Liter. Hong Kong, 3.44, 3.46, May/25, USD/Liter. Related. Nigeria Cuts Gasoline Prices · Zimbabwe Hikes Gasoline Prices by 47 ...', 'The current gasoline price in Singapore is SGD 2.74 per liter or USD 2.14 per liter and was updated on 23-Jun-2025. For comparison, the average price of ...', 'Singapore Gasoline 92# Unlea

In [134]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "what is the the gasoline price in Mumbai today?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `google_search` with `Mumbai gasoline price November 2 2023`


['Whereas in Mumbai, petrol is available at Rs 106.31 and diesel at Rs 94.27 per litre. While petrol in Kolkata is Rs 106.03 and diesel is Rs ...', 'Petrol and Diesel prices today, November 2, 2023: Petrol price in Hyderabad is Rs. 119.49 per litre, and diesel price at Rs. 105.65 per litre.', '... Mumbai, Chennai, and Kolkata, for the month of November 2023. The prices remained consistent throughout the month, with Delhi at ₹89.62, Mumbai at ₹94.27 ...', 'In Mumbai, petrol is being sold for Rs 106.31/litre and diesel for Rs 94.27/litre. ; Citywide breakdown ; Delhi, petrol and diesel cost Rs 96.72/ ...', 'In Mumbai, petrol demands a higher price of Rs 106.31 per liter, with diesel following suit at Rs 94.27 per litre. In Kolkata, the cost of ...', 'Pump Prices (INR/LTR) ; HUBLI, 91.61, 101.61 ; MUMBAI, 90.52, 100.52 ; JAMNAGAR, 90.36, 100.36 ; BHUJ, 90.8, 100.8 ...', 'Stay 

In [135]:
# Invoke the agent executor with a sample input
response = agent_executor.invoke({"input": "Where is Bird Paradise located exactly in Singapore as of today June 2025?"})

# Print the response
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `google_search` with `Bird Paradise location Singapore June 2025`


['Limited parking on 28 Jun 2025. Please use alternative travel options. 3 of 4: Egg Splash Wet Play will be closed for renovation from 14 April 2025. 4 of 4 ...', "Singapore's new bird park is open! Bird Paradise is located at Mandai Wildlife Reserve, next to Night Safari, River Wonders and Singapore Zoo.", 'What to know about Bird Paradise from 2025 Bird Paradise ticket prices/promos to feeding times, bird shows, and glamping with penguins.', 'Bird Paradise Singapore is located at 11 Lorong Lada Hitam, Singapore 778725. Which city is Bird Paradise Singapore in?', 'Pay a visit to the new Bird Paradise in Mandai, Singapore, and be immersed into a naturalistic and colourful world, home to 3,500 birds from over 400 ...', 'Bird Paradise Singapore is a world-renowned bird sanctuary. The sanctuary is a popular tourist attraction, drawing visitors from all over the world.', 

# Happy Learning